In [ ]:
import os, random, glob, json, math
from pathlib import Path

IMAGES_DIR = "/kaggle/input/datasets/dimabimov/aaa-projec/dataset/images"
if not os.path.isdir(IMAGES_DIR):  # путь монтирования может отличаться, ищем папку images
    cands = sorted(glob.glob("/kaggle/input/**/images", recursive=True))
    if cands:
        IMAGES_DIR = cands[0]
print("IMAGES_DIR:", IMAGES_DIR, "| exists:", os.path.isdir(IMAGES_DIR))

OUT_DIR = Path("/kaggle/working")
QC_DIR = OUT_DIR / "qc"; QC_DIR.mkdir(parents=True, exist_ok=True)
LABELS_CSV = OUT_DIR / "siglip2_labels.csv"
EMB_PATH = OUT_DIR / "siglip2_image_embeddings.npy"

SIGLIP_MODEL = "google/siglip2-so400m-patch14-384"
CLASSES = ["real_estate", "floor_plan", "screenshot"]

SEED = 42
SAMPLE_SIZE = 40000      # None = разметить все 100k (рекомендуется для финального прогона)
ZS_BATCH = 32            # батч zero-shot инференса SigLIP2
CONF_THRESHOLD = 0.85    # мин. softmax-уверенность псевдо-лейбла, чтобы взять его в трейн
MAX_PER_CLASS = 4000     # потолок на класс при балансировке трейн-сета
IMG_SIZE = 224           # вход MobileNetV3

random.seed(SEED)


In [ ]:
!pip -q install -U "transformers>=4.49" timm onnx onnxruntime scikit-learn 2>/dev/null
import transformers, torch
print("transformers", transformers.__version__, "| torch", torch.__version__, "| cuda", torch.cuda.is_available())


In [ ]:
EXTS = (".jpg", ".jpeg", ".png", ".webp", ".bmp")
all_paths = [p for p in glob.glob(os.path.join(IMAGES_DIR, "**", "*"), recursive=True)
             if p.lower().endswith(EXTS)]
print("всего изображений:", len(all_paths))

if SAMPLE_SIZE and len(all_paths) > SAMPLE_SIZE:
    paths = random.sample(all_paths, SAMPLE_SIZE)
else:
    paths = list(all_paths)
print("будет размечено:", len(paths))


In [ ]:
PROMPTS = {
    "real_estate": [
        "a real estate listing photo of an apartment interior",
        "a photograph of a room inside a home",
        "a photo of the exterior of a house or building",
        "a photo of a garage, warehouse or industrial space",
        "a photo of a land plot, yard or field",
        "a real estate photograph of a property",
    ],
    "floor_plan": [
        "an architectural floor plan drawing",
        "a 2D apartment layout plan with rooms and walls",
        "a blueprint schematic of a building floor",
        "a black and white technical floor plan on a white background",
    ],
    "screenshot": [
        "a mobile phone screenshot with black bars at the top and bottom",
        "a smartphone app screenshot with a status bar and black letterbox borders",
        "a screen capture from a phone showing an image with black margins",
        "a vertical phone screenshot of a real estate app interface",
    ],
}


In [ ]:
from transformers import AutoModel, AutoProcessor

device = "cuda"
dtype = torch.float16
model = AutoModel.from_pretrained(SIGLIP_MODEL, torch_dtype=dtype).to(device).eval()
processor = AutoProcessor.from_pretrained(SIGLIP_MODEL)

flat_prompts, owner = [], []
for ci, c in enumerate(CLASSES):
    for t in PROMPTS[c]:
        flat_prompts.append(t)
        owner.append(ci)
owner = torch.tensor(owner, device=device)

with torch.no_grad():
    tin = processor(text=flat_prompts, padding="max_length", max_length=64, return_tensors="pt").to(device)
    tfeat = model.get_text_features(**tin)
    tfeat = (tfeat / tfeat.norm(dim=-1, keepdim=True)).float()

logit_scale = model.logit_scale.exp().float()
logit_bias = model.logit_bias.float()
print("prompts:", len(flat_prompts), "| text emb dim:", tfeat.shape[1])


In [ ]:
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

Image.MAX_IMAGE_PIXELS = None
_size = processor.image_processor.size
_HW = _size.get("height", _size.get("shortest_edge", 384))

class ZSDataset(Dataset):
    def __init__(self, paths):
        self.paths = paths
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, i):
        try:
            img = Image.open(self.paths[i]).convert("RGB")
            px = processor(images=img, return_tensors="pt")["pixel_values"][0]
            return px, i, 1
        except Exception:
            return torch.zeros(3, _HW, _HW), i, 0

loader = DataLoader(ZSDataset(paths), batch_size=ZS_BATCH, num_workers=4, pin_memory=True)

n, nc = len(paths), len(CLASSES)
class_logits = np.zeros((n, nc), dtype=np.float32)
emb_store = np.zeros((n, tfeat.shape[1]), dtype=np.float16)
oks = np.zeros(n, dtype=np.int8)

with torch.no_grad():
    for px, idx, ok in tqdm(loader, desc="zero-shot"):
        px = px.to(device, dtype=dtype, non_blocking=True)
        ifeat = model.get_image_features(pixel_values=px)
        ifeat = (ifeat / ifeat.norm(dim=-1, keepdim=True)).float()
        logits = ifeat @ tfeat.t() * logit_scale + logit_bias        # [B, num_prompts]
        agg = torch.stack([logits[:, owner == ci].mean(dim=1) for ci in range(nc)], dim=1)
        ii = idx.numpy()
        class_logits[ii] = agg.cpu().numpy()
        emb_store[ii] = ifeat.cpu().numpy().astype(np.float16)
        oks[ii] = ok.numpy().astype(np.int8)


In [ ]:
import pandas as pd

probs = torch.softmax(torch.from_numpy(class_logits), dim=1).numpy()
pred = probs.argmax(1)
conf = probs.max(1)

df = pd.DataFrame({
    "path": paths,
    "filename": [os.path.basename(p) for p in paths],
    "pred_class": [CLASSES[i] for i in pred],
    "confidence": conf,
    "ok": oks,
})
for ci, c in enumerate(CLASSES):
    df[f"p_{c}"] = probs[:, ci]

df.to_csv(LABELS_CSV, index=False)
np.save(EMB_PATH, emb_store)   # пригодятся, если захочешь голову на эмбеддингах для сверки

print("распределение классов (все):")
print(df["pred_class"].value_counts())
print("\nуверенных (conf >= %.2f):" % CONF_THRESHOLD)
print(df[df.confidence >= CONF_THRESHOLD]["pred_class"].value_counts())


In [ ]:
import matplotlib.pyplot as plt

def gallery(c, n=25, only_conf=True):
    sub = df[(df.pred_class == c) & (df.ok == 1)]
    if only_conf:
        sub = sub[sub.confidence >= CONF_THRESHOLD]
    if len(sub) == 0:
        print("нет примеров для", c); return
    sub = sub.sample(min(n, len(sub)), random_state=SEED)
    cols = 5; rows = math.ceil(len(sub) / cols)
    plt.figure(figsize=(cols * 2.4, rows * 2.4))
    for k, (_, r) in enumerate(sub.iterrows()):
        plt.subplot(rows, cols, k + 1)
        try:
            plt.imshow(Image.open(r.path).convert("RGB"))
        except Exception:
            pass
        plt.title(f"{r.confidence:.2f}", fontsize=8)
        plt.axis("off")
    plt.suptitle(f"{c} (n_total={len(df[df.pred_class==c])})")
    plt.tight_layout()
    plt.savefig(QC_DIR / f"qc_{c}.png", dpi=90)
    plt.show()

for c in CLASSES:
    gallery(c)


In [ ]:
import cv2

def letterbox_score(path):
    """Доля верх+низ краёв (по 15%), где медиана яркости строки <= 8 (чёрная полоса)."""
    g = cv2.imread(path, cv2.IMREAD_GRAYSCALE)
    if g is None:
        return 0.0
    h = g.shape[0]
    band = max(1, int(h * 0.15))
    med = np.median(g, axis=1)
    top = (med[:band] <= 8).mean()
    bot = (med[-band:] <= 8).mean()
    return float(max(top, bot))

sample = df[df.ok == 1].sample(min(2000, len(df)), random_state=SEED).copy()
sample["lb"] = sample["path"].map(letterbox_score)

missed = sample[(sample.lb > 0.6) & (sample.pred_class != "screenshot")]
falsepos = sample[(sample.lb < 0.2) & (sample.pred_class == "screenshot")]
print("есть полосы, но НЕ screenshot:", len(missed))
print("назван screenshot, но полос нет:", len(falsepos))
print("\nдоля screenshot-предсказаний с полосами:",
      round((sample[sample.pred_class=="screenshot"].lb > 0.4).mean(), 3) if (sample.pred_class=="screenshot").any() else None)


In [ ]:
from sklearn.model_selection import train_test_split

use = df[(df.ok == 1) & (df.confidence >= CONF_THRESHOLD)]
parts = []
for c in CLASSES:
    s = use[use.pred_class == c]
    parts.append(s.sample(min(len(s), MAX_PER_CLASS), random_state=SEED))
train_df = pd.concat(parts).reset_index(drop=True)
print(train_df.pred_class.value_counts())

tr, va = train_test_split(train_df, test_size=0.15, stratify=train_df.pred_class, random_state=SEED)
print("train:", len(tr), "| val:", len(va))


In [ ]:
import torchvision.transforms as T

cls2idx = {c: i for i, c in enumerate(CLASSES)}
_norm = T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
train_tf = T.Compose([
    T.Resize((IMG_SIZE, IMG_SIZE)),
    T.RandomHorizontalFlip(),
    T.ColorJitter(0.2, 0.2, 0.1, 0.0),
    T.ToTensor(), _norm,
])
val_tf = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE)), T.ToTensor(), _norm])

class ClsDataset(Dataset):
    def __init__(self, frame, tf):
        self.f = frame.reset_index(drop=True)
        self.tf = tf
    def __len__(self):
        return len(self.f)
    def __getitem__(self, i):
        r = self.f.iloc[i]
        img = Image.open(r["path"]).convert("RGB")
        return self.tf(img), cls2idx[r["pred_class"]]

from torch.utils.data import WeightedRandomSampler
counts = tr.pred_class.value_counts().to_dict()
weights = tr.pred_class.map(lambda c: 1.0 / counts[c]).values
sampler = WeightedRandomSampler(torch.DoubleTensor(weights), num_samples=len(weights), replacement=True)

train_loader = DataLoader(ClsDataset(tr, train_tf), batch_size=64, sampler=sampler, num_workers=4, pin_memory=True)
val_loader = DataLoader(ClsDataset(va, val_tf), batch_size=64, shuffle=False, num_workers=4, pin_memory=True)


In [ ]:
import timm
from torch.cuda.amp import autocast, GradScaler

net = timm.create_model("mobilenetv3_small_100", pretrained=True, num_classes=len(CLASSES)).to(device)
EPOCHS = 12
opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=EPOCHS * len(train_loader))
crit = torch.nn.CrossEntropyLoss(label_smoothing=0.05)
scaler = GradScaler()

@torch.no_grad()
def evaluate():
    net.eval()
    ys, ps = [], []
    for x, y in val_loader:
        x = x.to(device, non_blocking=True)
        with autocast():
            out = net(x)
        ps.append(out.argmax(1).cpu()); ys.append(y)
    import torch as _t
    return _t.cat(ys).numpy(), _t.cat(ps).numpy()

for ep in range(EPOCHS):
    net.train()
    run = 0.0
    for x, y in tqdm(train_loader, desc=f"epoch {ep+1}/{EPOCHS}"):
        x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True)
        opt.zero_grad()
        with autocast():
            loss = crit(net(x), y)
        scaler.scale(loss).backward()
        scaler.step(opt); scaler.update(); sched.step()
        run += loss.item() * x.size(0)
    yv, pv = evaluate()
    acc = (yv == pv).mean()
    print(f"epoch {ep+1}: train_loss={run/len(tr):.4f}  val_acc={acc:.4f}")


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

yv, pv = evaluate()
print(classification_report(yv, pv, target_names=CLASSES, digits=3))
cm = confusion_matrix(yv, pv)
print("confusion matrix (rows=true SigLIP, cols=pred CNN):")
print(pd.DataFrame(cm, index=CLASSES, columns=CLASSES))


In [ ]:
net.eval()
ckpt = {
    "state_dict": net.state_dict(),
    "arch": "mobilenetv3_small_100",
    "classes": CLASSES,
    "img_size": IMG_SIZE,
    "normalize": {"mean": [0.485, 0.456, 0.406], "std": [0.229, 0.224, 0.225]},
}
torch.save(ckpt, OUT_DIR / "photo_type_classifier.pth")

dummy = torch.randn(1, 3, IMG_SIZE, IMG_SIZE, device=device)
torch.onnx.export(
    net, dummy, str(OUT_DIR / "photo_type_classifier.onnx"),
    input_names=["input"], output_names=["logits"],
    dynamic_axes={"input": {0: "batch"}, "logits": {0: "batch"}},
    opset_version=17,
)
json.dump(
    {"arch": "mobilenetv3_small_100", "classes": CLASSES, "img_size": IMG_SIZE,
     "normalize": ckpt["normalize"]},
    open(OUT_DIR / "photo_type_classifier.json", "w"), indent=2,
)
print("сохранено:", [p.name for p in OUT_DIR.glob("photo_type_classifier.*")])
